# 02 – Veri Bölme, Baseline'lar ve İlk Model
**Hafta:** H5–H6 (16.11 – 29.11) · **Puanlama:** Baseline ve model karşılaştırması %25 · **Slaytlar:** 33–35, 39, 44–48, 56–60

**Amaç:** Slayttaki iş akışını ilk kez baştan sona uygulamak: iki baseline ve lojistik regresyonu **doğrulama setinde** değerlendirmek.

**Bu notebook'un iş akışı (slayt 53, 56):**
`veri` → `hazırla (X, y, bölme)` → `model = ...` → `fit()` → `predict()` → `metrik` → `yorum`

> **Test setine bu notebook'ta dokunulmaz.** `X_test` ve `y_test` sadece `06_final_test`'te oluşturulur.

### Adım 0: Kurulum
*Slayt 55*

**Ne yapıyoruz:** Kütüphaneleri ve proje kodunu yüklüyoruz.

In [ ]:
# proje kökünü Python yoluna ekle (src/ klasörünü kullanabilmek için)
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score)

# proje kodu
from src import config
from src.data import load_raw, clean, split_xy, random_split, time_split, describe_split
from src.features import make_preprocessor, drop_cols
from src.models import PageValuesRule
from src.evaluate import scores, best_threshold, results_table, plot_pr_curves

### Adım 1: Problemi ve metriği hatırla
*Slayt 8, 17, 44, 47–48*

**Ne yapıyoruz:** Kod yazmadan önce neyi, neden ölçtüğümüzü ve hangi hatanın daha pahalı olduğunu yazıyoruz (slayt 44: "Sadece accuracy'yi sorma, hangi hatalar önemli diye sor").

| | |
|---|---|
| **Hedef (y)** | `Revenue` (1 = alışveriş yaptı) |
| **Görev tipi** | İkili sınıflandırma |
| **Sınıf dağılımı** | Temiz veride %15,6 alır, %84,4 almaz (01_eda, Adım 10) |
| **Ana metrik** | PR-AUC: eşikten bağımsız, azınlık sınıfa (alıcılara) odaklanır. Rastgele bir modelin PR-AUC'u ≈ pozitif oranı (≈ 0,156) |
| **Karar metriği** | F1: eşik, doğrulama setinde F1'i en yüksek yapan değer olarak seçilir |
| **Yanında raporlananlar** | Precision, recall; accuracy sadece yanında, tek başına asla |

#### Hangi hata daha pahalı? (slayt 44, 47, 48)
**Kullanım senaryosu:** Modelin "alır" dediği oturumlara siteden çıkmadan önce bir kampanya (indirim, ücretsiz kargo) gösterilir.

| Hata | Ne oldu? | Sonucu |
|---|---|---|
| **Yanlış pozitif (FP)** | Almayacak birine "alır" dedik | Boşuna kampanya gösterildi. Kampanya kullanılmazsa maliyeti küçük: reklam alanı, kullanıcıyı biraz rahatsız etmek |
| **Yanlış negatif (FN)** | Alacak birine "almaz" dedik | Alıcı adayı kampanyayı görmedi; satış kaçabilir |

**Sonuç:** Bu senaryoda yanlış negatif genelde daha pahalı, yani **recall** biraz daha önemli.

**Neden yine de F1?** Gerçek maliyet rakamlarımız yok: kampanyanın maliyeti ve ortalama sepet tutarı veride bulunmuyor. Maliyetleri uyduramayacağımız için eşiği iki hatayı dengeleyen **F1** ile seçiyoruz (öneri formundaki plan). Eşiğin bir karar olduğunu unutmuyoruz (slayt 48): düşük eşik → daha çok alıcı yakalanır (recall ↑) ama daha çok yanlış alarm olur (precision ↓). Bu dengeyi Adım 9'da ve 04_deneyler'de ayrıca göstereceğiz.

**Kontrol soruları:**
- Slayt 48'deki kanser taraması örneğinde eşik düşük mü olmalı, yüksek mi? Bizim problemde?
- Gerçek maliyetleri bilseydik eşiği nasıl seçerdik?

### Adım 2: Veriyi yükle ve temizle
*Slayt 32, 53*

**Ne yapıyoruz:** Ham veriyi okuyup 01'de verdiğimiz temizlik kararını uyguluyoruz.

**Sözde kod (slayt 57):**
```
ham = dosyayı oku
df = temizle(ham)   # tekrar eden satırları sil, True/False -> 0/1
```

In [ ]:
# veriyi yükle
raw = load_raw()

# temizle (bölmeden ÖNCE)
df = clean(raw)

### Adım 3: X ve y'yi seç
*Slayt 15–16*

**Ne yapıyoruz:** Hangi sütunların girdi (X), hangisinin hedef (y) olduğunu belirliyoruz.

**Sözde kod (slayt 57):**
```
X = 10 sayısal + 7 kategorik sütun
y = Revenue
```

In [ ]:
# girdi sütunları (X)
print("Sayısal   :", config.NUM_COLS)
print("Kategorik :", config.CAT_COLS)

# hedef sütun (y)
print("Hedef     :", config.TARGET)

### Adım 4: Veriyi böl (ön işlemeden ÖNCE)
*Slayt 33–35*

**Ne yapıyoruz:** Veriyi eğitim / doğrulama / test olarak ayırıyoruz. İki bölme kuruyoruz: rastgele (tabakalı) ve zamana göre.

**Sözde kod (slayt 57):**
```
eğitim, doğrulama, test = rastgele_böl(df)      # %70 / %15 / %15
X_train, y_train = eğitimden X ve y
X_val, y_val     = doğrulamadan X ve y
(test setine dokunma)
```

**Kontrol soruları:**
- Neden ön işlemeden önce bölüyoruz? (slayt 61)
- Zamana göre bölmede eğitim ve testin alışveriş oranları neden farklı?

In [ ]:
# böl
train, val, test = random_split(df)
t_train, t_val, t_test = time_split(df)

# her parçanın boyutu ve alışveriş oranı (slayt 51)
pd.concat([
    describe_split("rastgele", (train, val, test)),
    describe_split("zaman", (t_train, t_val, t_test)),
])

In [ ]:
# X ve y: eğitim ve doğrulama (test setine DOKUNMUYORUZ)
X_train, y_train = split_xy(train)
X_val, y_val = split_xy(val)

print("X_train:", X_train.shape, "| X_val:", X_val.shape)

**Yorum:** 

### Adım 5: Baseline 1: herkese "almaz" (tam örnek)
*Slayt 39, 46, 56, 59*

**Ne yapıyoruz:** En basit referans: her oturuma çoğunluk sınıfını (almaz) söyleyen model. Bu adım slayt 59'daki kalıbın tam örneği; sonraki modeller aynı kalıpla yazılır.

**Sözde kod (slayt 57):**
```
model = çoğunluk sınıfını söyleyen model
model eğitim verisinden öğrenir
tahminler = model(doğrulama)
skor = karşılaştır(tahminler, gerçek)
```

In [ ]:
# modeli oluştur
model = DummyClassifier(strategy="most_frequent")

# eğit (sadece eğitim verisi)
model.fit(X_train, y_train)

# tahmin et (doğrulama verisi)
y_pred = model.predict(X_val)
p_val = model.predict_proba(X_val)[:, 1]   # "alır" olasılığı

# değerlendir
print("Accuracy :", round(accuracy_score(y_val, y_pred), 3))
print("Precision:", round(precision_score(y_val, y_pred, zero_division=0), 3))
print("Recall   :", round(recall_score(y_val, y_pred), 3))
print("F1       :", round(f1_score(y_val, y_pred), 3))
print("PR-AUC   :", round(average_precision_score(y_val, p_val), 3))

**Yorum:** Accuracy %84 civarında ama recall 0: model hiçbir alıcıyı yakalamıyor. Dengesiz veride accuracy bu yüzden yanıltıcı (slayt 46).

In [ ]:
# sonuçları bir sözlükte topluyoruz. scores() yukarıdaki 5 metriği aynı şekilde hesaplar.
results = {}
probas = {}

results["Baseline: herkese almaz"] = scores(y_val, p_val, threshold=0.5)
probas["Baseline: herkese almaz"] = p_val

results_table(results)

### Adım 6: Baseline 2: "PageValues > 0 ise alır" kuralı
*Slayt 39*

**Ne yapıyoruz:** Alan bilgisine dayalı basit kural. Öğrenme yapmaz ama aynı kalıpla değerlendirilir.

**Sözde kod (slayt 57):**
```
model = kural (PageValues > 0 ise alır)
tahminler = model(doğrulama)
skor = karşılaştır(tahminler, gerçek)
```

**Kontrol soruları:**
- Bir ML modeli bu kuralı geçemezse ne anlama gelir? (slayt 39)

In [ ]:
# modeli oluştur

# eğit (kural öğrenme yapmaz ama fit çağrısı aynı kalıp için var)

# tahmin et (doğrulama verisi)

# değerlendir ve results sözlüğüne ekle

**Yorum:** 

### Adım 7: Ön işleme: Pipeline
*Slayt 24–25, 61*

**Ne yapıyoruz:** Lojistik regresyon için sayısal sütunları ölçekleyip kategorik sütunları one-hot kodluyoruz. Bu adımlar Pipeline'ın içine konur; böylece sadece eğitim verisinden öğrenirler.

**Sözde kod (slayt 57):**
```
ön_işleme = [sayısal sütunları ölçekle, kategorikleri one-hot yap]
model = Pipeline(ön_işleme -> sınıflandırıcı)
model.fit(eğitim)   # ölçekleyici de SADECE eğitimden öğrenir
```

**Kontrol soruları:**
- Ölçekleyiciyi tüm veriye fit etseydik ne olurdu? (slayt 61)

In [ ]:
# ön işleme nesnesi (henüz fit edilmedi): src/features.py → make_preprocessor
preprocessor = make_preprocessor()
preprocessor

### Adım 8: Lojistik regresyon
*Slayt 56, 59*

**Ne yapıyoruz:** İlk gerçek model. Önce varsayılan eşik 0,5 ile değerlendiriyoruz.

**Sözde kod (slayt 57):**
```
model = Pipeline(ön_işleme -> LogisticRegression)
model eğitim verisinden öğrenir
olasılıklar = model(doğrulama)
skor = karşılaştır(olasılıklar, gerçek)
```

**Kontrol soruları:**
- `class_weight="balanced"` ne işe yarar? Kullanınca ve kullanmayınca ne değişiyor? (bir seferde tek değişiklik, slayt 42)

In [ ]:
# modeli oluştur (ön işleme + lojistik regresyon)

# eğit (sadece eğitim verisi)

# tahmin et (doğrulama verisi): "alır" olasılıkları

# değerlendir (eşik 0.5) ve results sözlüğüne ekle

**Yorum:** 

### Adım 9: Eşik seçimi
*Slayt 48*

**Ne yapıyoruz:** 0,5 eşiği zorunlu değil. Doğrulama setinde F1'i en yüksek yapan eşiği seçiyoruz.

**Sözde kod (slayt 57):**
```
eşik = doğrulamada F1'i en yüksek yapan değer
skor = karşılaştır(olasılık >= eşik, gerçek)
```

**Kontrol soruları:**
- Eşiği neden test setinde seçmiyoruz? (slayt 34)

In [ ]:
# en iyi eşik (sadece doğrulama verisiyle)

# bu eşikle değerlendir ve results sözlüğüne ekle

**Yorum:** 

### Adım 10: Karşılaştırma tablosu ve PR eğrisi
*Slayt 39, 47*

**Ne yapıyoruz:** İki baseline ve lojistik regresyonu yan yana koyuyoruz.

**Sözde kod (slayt 57):**
```
tablo = tüm sonuçlar
PR eğrilerini çiz, reports/figures/02_pr_baseline.png olarak kaydet
```

In [ ]:
# tablo

# PR eğrisi

**Yorum:** 

### Slayt 60 soruları
Bu notebook'taki koda bakarak cevaplayın (hoca bu soruları soracak):

| Soru | Cevap |
|---|---|
| X ve y nerede tanımlandı? | |
| Bölme nerede yapıldı? Ön işlemeden önce mi? | |
| `fit()` hangi veriyi kullanıyor? | |
| `predict()` hangi veriyi kullanıyor? | |
| Hangi metrik hesaplanıyor, neden o? | |
| Accuracy bu veride neden yanıltıcı? | |

## Bulgular ve kararlar
> Bu hücreyi **kendi cümlelerinizle** yazın. Sunumda ve hocanın kontrolünde bunu savunacaksınız.

- Bulgu 1:
- Bulgu 2:
- Bu bulgulara göre aldığımız karar(lar):
- Deney günlüğüne yazıldı mı? (`reports/deney_gunlugu.md`)
- Git commit atıldı mı?
- Lojistik regresyon kural baseline'ını ne kadar geçiyor? Bu fark anlamlı mı?